# 01 · Análisis exploratorio (EDA) — Covertype, grupo 2

**Objetivo:** entender los datos que recolectó el DAG `covertype_ingest` antes de entrenar.

| Sección | Pregunta | Fuente |
|---|---|---|
| 1. Trazabilidad | ¿Qué peticiones se hicieron, a qué grupo y qué batch respondió la API? | `raw.fetch_log` |
| 2. Calidad por run | ¿Cuántas filas fueron nuevas, duplicadas o inválidas en cada run? | `processed.run_stats` |
| 3. Cuarentena | ¿Qué filas se rechazaron y por qué? | `processed.rejected` |
| 4. Variable objetivo | ¿Qué clases hay y qué tan desbalanceadas están? | `train.covertype` |
| 5. Categóricas | ¿Qué zonas y tipos de suelo aparecen? | `train.covertype` |
| 6. Numéricas | ¿Qué rangos tienen y cuáles separan mejor las clases? | `train.covertype` |

> Conexión con el usuario **`covertype_reader` (solo lectura)**: este notebook no puede modificar los datos del pipeline.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

engine = create_engine(os.environ["DATA_DB_URL"])


def q(sql: str) -> pd.DataFrame:
    # Ejecuta una consulta de solo lectura y devuelve un DataFrame
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)


NOMBRES_CLASE = {0: "Spruce/Fir", 1: "Lodgepole Pine", 2: "Ponderosa Pine", 3: "Cottonwood/Willow",
                 4: "Aspen", 5: "Douglas-fir", 6: "Krummholz"}
print("Conectado a:", engine.url.render_as_string(hide_password=True))

## 1. Trazabilidad de la recolección

Cada fila es **una ejecución del DAG = una petición a la API**. Es la evidencia de *grupo y batch* que pide el enunciado.

In [ ]:
fetch = q('''
    SELECT dag_run_id, requested_at, group_number, batch_number, http_status, rows_received, status
    FROM raw.fetch_log
    ORDER BY requested_at
''')
fetch["requested_at"] = fetch["requested_at"].dt.tz_convert("America/Bogota").dt.strftime("%Y-%m-%d %H:%M:%S")
fetch["batch_number"] = fetch["batch_number"].astype("Int64")   # entero con nulos (exhausted) en vez de 1.0, 2.0...
display(fetch)

ok = fetch[fetch.status == "ok"]
print(f"Peticiones: {len(fetch)} | ok: {len(ok)} | exhausted: {(fetch.status == 'exhausted').sum()} | "
      f"error: {(fetch.status == 'error').sum()}")
ok = ok.astype({"batch_number": int})
print(f"Grupos consultados: {sorted(int(g) for g in fetch.group_number.unique())}")
print(f"Batches distintos recolectados: {ok.batch_number.nunique()} -> {sorted(int(b) for b in ok.batch_number.unique())}")
repetidos = ok.batch_number.value_counts()
repetidos = repetidos[repetidos > 1]
if len(repetidos):
    print(f"Batches pedidos más de una vez (misma ventana de tiempo): {repetidos.to_dict()}")

## 2. Calidad de datos por run: nuevas vs. duplicadas vs. inválidas

Si la API entregara una porción **distinta** del dataset en cada batch, casi todas las filas serían nuevas.
Una proporción de duplicados **creciente** indica que los batches se muestrean del **mismo universo** de filas.

In [ ]:
stats = q('''
    SELECT f.requested_at, f.batch_number, s.rows_raw, s.rows_new, s.rows_duplicate, s.rows_invalid
    FROM processed.run_stats s JOIN raw.fetch_log f USING (dag_run_id)
    ORDER BY f.requested_at
''')
stats["pct_duplicadas"] = (100 * stats.rows_duplicate / stats.rows_raw).round(1)
stats["run"] = range(1, len(stats) + 1)
display(stats.drop(columns="requested_at"))

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(stats.run, stats.rows_new, label="nuevas", color="#2f7ed8")
ax1.bar(stats.run, stats.rows_duplicate, bottom=stats.rows_new, label="duplicadas", color="#c0c0c0")
ax1.set_xlabel("run (en orden de ejecución)")
ax1.set_ylabel("filas")
ax1.set_xticks(stats.run)
ax2 = ax1.twinx()
ax2.plot(stats.run, stats.pct_duplicadas, color="#d9534f", marker="o", label="% duplicadas")
ax2.set_ylabel("% duplicadas")
ax2.set_ylim(0, 100)
fig.legend(loc="upper left", bbox_to_anchor=(0.08, 0.95))
ax1.set_title("Filas nuevas vs. duplicadas por run")
plt.tight_layout()
plt.show()

tot = stats[["rows_raw", "rows_new", "rows_duplicate", "rows_invalid"]].sum()
print(f"Total crudas: {tot.rows_raw} | únicas: {tot.rows_new} | duplicadas: {tot.rows_duplicate} "
      f"({100 * tot.rows_duplicate / tot.rows_raw:.1f} %) | inválidas: {tot.rows_invalid}")

**Contraste con la hipótesis del bug.** Si cada petición toma una muestra aleatoria de `m` filas de un universo fijo de `N` filas,
tras `k` peticiones el número esperado de filas únicas es:

$$E[\text{únicas}] = N\left(1 - (1 - m/N)^k\right)$$

Con `N` = una décima del dataset completo (581 012 / 10) y `m` = filas por petición, la fórmula predice cuántas filas únicas deberíamos ver.
Si lo observado coincide, queda **confirmado con datos** que todos los batches salen de la misma décima.

In [ ]:
N_TOTAL = 581_012                  # filas del dataset Covertype completo
N = N_TOTAL // 10                  # una décima: el 'batch' del que siempre muestrea la API
m = int(stats.rows_raw.median())   # filas por petición
k = len(stats)                     # peticiones con datos
esperadas = N * (1 - (1 - m / N) ** k)
observadas = int(tot.rows_new)
print(f"N={N}, m={m}, k={k}")
print(f"Únicas esperadas si todos los batches salen de la misma décima: {esperadas:,.0f}")
diferencia = 100 * (observadas - esperadas) / esperadas
print(f"Únicas observadas: {observadas:,}  (diferencia {diferencia:+.2f} %)")
if abs(diferencia) <= 5:
    print("-> Coincide con la hipótesis: todas las peticiones muestrean la MISMA décima del dataset.")
else:
    print("-> No coincide: la API no muestrea siempre del mismo universo de N filas "
          "(o el CSV de la fuente no tiene 581 012 filas).")

## 3. Cuarentena: filas rechazadas por la validación

Ninguna fila se descarta sin dejar rastro: el DAG guarda en `processed.rejected` la fila original y el motivo.

In [ ]:
rechazos = q("SELECT reason, count(*) AS filas FROM processed.rejected GROUP BY 1 ORDER BY 2 DESC")
if rechazos.empty:
    print("Sin filas rechazadas: todas las filas recibidas pasaron la validación.")
else:
    display(rechazos)
    display(q("SELECT reason, batch_number, raw_row FROM processed.rejected LIMIT 5"))

## 4. Variable objetivo: `cover_type`

In [ ]:
df = q("SELECT * FROM train.covertype")
print(f"train.covertype: {df.shape[0]:,} filas x {df.shape[1]} columnas")

clases = df.cover_type.value_counts().sort_index()
tabla = pd.DataFrame({
    "especie": [NOMBRES_CLASE.get(c, "?") for c in clases.index],
    "filas": clases.values,
    "%": (100 * clases / clases.sum()).round(2).values,
}, index=clases.index.rename("cover_type"))
display(tabla)

ax = tabla["filas"].plot.bar(figsize=(6, 3.5), color="#2f7ed8")
ax.set_title("Distribución de clases")
ax.set_xticklabels([f"{c}\n{NOMBRES_CLASE.get(c, '?')}" for c in tabla.index], rotation=0)
for p, pct in zip(ax.patches, tabla["%"]):
    ax.annotate(f"{pct} %", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
plt.tight_layout()
plt.show()

mayoritaria = tabla["%"].max()
print(f"Clases presentes: {list(tabla.index)} de 7 posibles.")
print(f"Un modelo que siempre prediga la clase mayoritaria ya tendría {mayoritaria:.1f} % de accuracy "
      f"-> la accuracy NO sirve como métrica principal; usaremos F1-macro.")

## 5. Variables categóricas

In [ ]:
print("wilderness_area:")
display(df.wilderness_area.value_counts().to_frame("filas"))

suelos = df.soil_type.value_counts()
print(f"soil_type: {suelos.size} códigos distintos de 40 posibles")
ct = pd.crosstab(df.soil_type, df.cover_type).loc[suelos.index]
ax = ct.plot.barh(stacked=True, figsize=(8, 0.28 * len(ct) + 1.5), width=0.8)
ax.invert_yaxis()
ax.set_xlabel("filas")
ax.set_title("Tipo de suelo por clase")
ax.legend([f"{c} {NOMBRES_CLASE.get(c, '')}" for c in ct.columns], title="cover_type")
plt.tight_layout()
plt.show()

## 6. Variables numéricas

In [ ]:
NUMERICAS = ["elevation", "aspect", "slope", "horizontal_distance_to_hydrology",
             "vertical_distance_to_hydrology", "horizontal_distance_to_roadways",
             "hillshade_9am", "hillshade_noon", "hillshade_3pm", "horizontal_distance_to_fire_points"]
display(df[NUMERICAS].describe().T.round(1))

fig, axes = plt.subplots(2, 5, figsize=(15, 5.5))
for ax, col in zip(axes.flat, NUMERICAS):
    ax.hist(df[col], bins=40, color="#2f7ed8")
    ax.set_title(col, fontsize=9)
plt.suptitle("Distribuciones de las variables numéricas")
plt.tight_layout()
plt.show()

### ¿Qué variables separan las clases?

Comparamos cada variable por clase. La **elevación** debería dominar: el Spruce/Fir crece a mayor altura que el Lodgepole Pine.
El *effect size* (diferencia de medianas entre clases / desviación estándar global) resume qué tan separadas están.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, col in zip(axes.flat, NUMERICAS):
    grupos = [df.loc[df.cover_type == c, col] for c in clases.index]
    ax.boxplot(grupos, showfliers=False)
    ax.set_xticks(range(1, len(clases) + 1), [str(c) for c in clases.index])
    ax.set_title(col, fontsize=9)
plt.suptitle("Variables numéricas por clase (sin outliers)")
plt.tight_layout()
plt.show()

medianas = df.groupby("cover_type")[NUMERICAS].median()
separacion = ((medianas.max() - medianas.min()) / df[NUMERICAS].std()).sort_values(ascending=False)
display(separacion.round(2).to_frame("separación entre clases (medianas / std)"))

## 7. Resumen automático del EDA

In [ ]:
resumen = {
    "peticiones_ok": int(len(ok)),
    "batches_distintos": int(ok.batch_number.nunique()),
    "filas_unicas_train": int(len(df)),
    "pct_duplicadas_total": round(100 * tot.rows_duplicate / tot.rows_raw, 1),
    "filas_en_cuarentena": int(rechazos.filas.sum()) if not rechazos.empty else 0,
    "clases_presentes": [int(c) for c in clases.index],
    "pct_clase_mayoritaria": float(mayoritaria),
    "pct_clase_minoritaria": float(tabla["%"].min()),
    "zonas": sorted(df.wilderness_area.unique().tolist()),
    "suelos_distintos": int(suelos.size),
    "variable_mas_separadora": separacion.index[0],
}
for k, v in resumen.items():
    print(f"{k:28s} {v}")